# 🚀 BTC 机器学习量化交易策略

## 基于 XGBoost + LSTM 集成学习的比特币量化交易系统

---

### 📋 目录

1. [环境初始化与配置](#1-环境初始化与配置)
2. [数据获取](#2-数据获取)
3. [数据预处理与探索性分析](#3-数据预处理与探索性分析)
4. [特征工程](#4-特征工程)
5. [模型训练与评估 - XGBoost](#5-模型训练与评估---xgboost)
6. [模型训练与评估 - LSTM](#6-模型训练与评估---lstm)
7. [模型集成与策略设计](#7-模型集成与策略设计)
8. [回测执行](#8-回测执行)
9. [回测结果分析与投资建议](#9-回测结果分析与投资建议)
10. [实盘部署指南](#10-实盘部署指南)

---

**策略概述**：使用 XGBoost 和 LSTM 双模型集成预测 BTC/USDT 4小时K线的价格方向，通过加权投票融合信号，结合 Kelly 公式仓位管理和 ATR 动态风控执行交易。

| 参数 | 值 |
|------|-----|
| 交易标的 | BTC/USDT |
| 时间周期 | 4h |
| 数据范围 | 2020-01 至今 |
| 初始资金 | 10,000 USDT |
| 手续费 | 0.1% |
| 滑点 | 0.05% |

## 1. 环境初始化与配置

本节完成以下工作：
- 检查 Python 环境版本
- 安装/验证所有依赖包
- 设置全局配置参数
- 初始化项目路径

In [1]:
# ============================================
# 1.1 环境检查与依赖安装
# ============================================
import sys
import os

print(f"Python 版本: {sys.version}")
print(f"工作目录: {os.getcwd()}")

# 检查 Python 版本（需要 3.8+）
assert sys.version_info >= (3, 8), "需要 Python 3.8 或更高版本"
print("✅ Python 版本检查通过")

# 安装依赖（如果尚未安装）
# 取消下面的注释来安装依赖
# !pip install -r requirements.txt -q

Python 版本: 3.12.13 (main, May 10 2026, 19:20:41) [Clang 22.1.3 ]
工作目录: /Users/Alexander_Xu/Study/Blockchain/Projects/Quant_BTC
✅ Python 版本检查通过


In [2]:
# ============================================
# 1.2 导入核心库并验证版本
# ============================================
import warnings
warnings.filterwarnings('ignore')

# 数据处理
import pandas as pd
import numpy as np

# 可视化
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

# 机器学习
import sklearn
import xgboost as xgb

# 深度学习
import tensorflow as tf

# 统计分析
import scipy
import statsmodels

# 交互组件
import ipywidgets as widgets
from tqdm.notebook import tqdm

# 数据获取
import ccxt

# 配置管理
from dotenv import load_dotenv

# 打印版本信息
print("📦 核心库版本信息：")
print(f"  pandas:      {pd.__version__}")
print(f"  numpy:       {np.__version__}")
print(f"  scikit-learn:{sklearn.__version__}")
print(f"  xgboost:     {xgb.__version__}")
print(f"  tensorflow:  {tf.__version__}")
print(f"  plotly:      {go.__version__ if hasattr(go, '__version__') else 'OK'}")
print(f"  ccxt:        {ccxt.__version__}")
print("\n✅ 所有依赖库导入成功")

📦 核心库版本信息：
  pandas:      2.3.3
  numpy:       2.2.6
  scikit-learn:1.8.0
  xgboost:     3.2.0
  tensorflow:  2.21.0
  plotly:      OK
  ccxt:        4.5.56

✅ 所有依赖库导入成功


In [3]:
# ============================================
# 1.3 全局配置参数
# ============================================

# 加载环境变量
load_dotenv()

# 项目路径配置
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')

# 确保目录存在
for dir_path in [DATA_RAW_DIR, DATA_PROCESSED_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# 添加 src 到 Python 路径
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# ==================== 策略全局配置 ====================
CONFIG = {
    # --- 交易标的与数据 ---
    'symbol': 'BTC/USDT',           # 交易对
    'timeframe': '4h',              # 主时间周期
    'start_date': '2020-01-01',     # 数据起始日期
    'exchange_id': 'binance',       # 交易所
    
    # --- 回测参数 ---
    'initial_capital': 10000,       # 初始资金 (USDT)
    'commission_rate': 0.001,       # 手续费率 0.1%
    'slippage_rate': 0.0005,        # 滑点率 0.05%
    
    # --- 模型参数 ---
    'train_ratio': 0.70,            # 训练集比例
    'val_ratio': 0.15,              # 验证集比例
    'test_ratio': 0.15,             # 测试集比例
    'lstm_window_size': 24,         # LSTM 滑动窗口大小（24个4h = 4天）
    'xgb_n_trials': 50,            # XGBoost Optuna 优化轮数
    
    # --- 策略参数 ---
    'long_threshold': 0.6,          # 做多信号阈值
    'short_threshold': 0.4,         # 做空信号阈值
    'stop_loss_atr_mult': 2.0,      # 止损 ATR 倍数
    'take_profit_atr_mult': 3.0,    # 止盈 ATR 倍数
    'max_holding_periods': 30,      # 最大持仓周期数（30个4h = 5天）
    'trailing_stop_atr_mult': 1.5,  # 移动止损 ATR 倍数
    
    # --- 集成模型权重 ---
    'xgb_weight': 0.5,              # XGBoost 权重
    'lstm_weight': 0.5,             # LSTM 权重
    
    # --- 实盘配置 ---
    'trading_mode': os.getenv('TRADING_MODE', 'backtest'),  # backtest/paper/live
    'api_key': os.getenv('BINANCE_API_KEY', ''),
    'api_secret': os.getenv('BINANCE_SECRET', ''),
}

# 打印配置摘要
print("⚙️ 全局配置参数：")
print(f"  交易标的:   {CONFIG['symbol']}")
print(f"  时间周期:   {CONFIG['timeframe']}")
print(f"  数据起始:   {CONFIG['start_date']}")
print(f"  初始资金:   {CONFIG['initial_capital']:,} USDT")
print(f"  手续费率:   {CONFIG['commission_rate']*100}%")
print(f"  滑点率:     {CONFIG['slippage_rate']*100}%")
print(f"  交易模式:   {CONFIG['trading_mode']}")
print(f"  做多阈值:   {CONFIG['long_threshold']}")
print(f"  做空阈值:   {CONFIG['short_threshold']}")
print("\n✅ 配置加载完成")

⚙️ 全局配置参数：
  交易标的:   BTC/USDT
  时间周期:   4h
  数据起始:   2020-01-01
  初始资金:   10,000 USDT
  手续费率:   0.1%
  滑点率:     0.05%
  交易模式:   backtest
  做多阈值:   0.6
  做空阈值:   0.4

✅ 配置加载完成


## 2. 数据获取

通过 ccxt 库连接 Binance 交易所，获取 BTC/USDT 从 2020年1月至今的 4小时 K线数据。

**数据字段说明**：
- `timestamp`: 时间戳
- `open`: 开盘价
- `high`: 最高价
- `low`: 最低价
- `close`: 收盘价
- `volume`: 成交量

**缓存机制**：首次获取后数据保存为本地 CSV，后续可直接从本地加载。

In [5]:
# ============================================
# 2.1 初始化数据获取器并下载数据
# ============================================
from data_fetcher import DataFetcher

# 创建数据获取器实例（代理通过 .env 中 PROXY_URL 配置，或直接传入）
fetcher = DataFetcher(
    data_dir=DATA_RAW_DIR,
    exchange_id=CONFIG['exchange_id'],
    proxy=os.getenv('PROXY_URL', None)
)

# 获取 BTC/USDT 4h K线数据（2020年至今）
df_raw = fetcher.fetch_ohlcv(
    symbol=CONFIG['symbol'],
    timeframe=CONFIG['timeframe'],
    start_date=CONFIG['start_date'],
    use_cache=True,
    update_cache=True
)

# 查看数据前几行
print(f'\n📋 数据预览（前5行）：')
df_raw.head()

  🌐 使用代理: http://127.0.0.1:12639
📊 获取 BTC/USDT 4h K线数据...
   起始日期: 2020-01-01
  🌐 开始从交易所下载数据（预估 14048 条）...



  ⚠️ 请求异常: binance GET https://api.binance.com/api/v3/exchangeInfo，等待后重试...


KeyboardInterrupt: 

## 3. 数据预处理与探索性分析

对原始 K 线数据进行清洗和探索性分析：
- 缺失值处理（前向填充）
- 异常值检测（Z-score / IQR 方法）
- 收益率分布分析
- 平稳性检验（ADF 检验）
- 自相关性分析（ACF/PACF）

In [ ]:
# ============================================
# 3.1 数据清洗
# ============================================
from preprocessor import DataPreprocessor, ExploratoryAnalyzer

# 初始化预处理器和分析器
preprocessor = DataPreprocessor(zscore_threshold=4.0)
analyzer = ExploratoryAnalyzer()

# 执行数据清洗
df_clean = preprocessor.clean(df_raw)

# 获取统计摘要
stats = preprocessor.get_summary_stats(df_clean)
preprocessor.print_summary_table(stats)

# 绘制交互式K线图（最近500根）
fig_candle = analyzer.plot_candlestick(df_clean, last_n=500)
fig_candle.show()

# 收益率分布分析
fig_dist = analyzer.plot_returns_distribution(df_clean)
fig_dist.show()

# 滚动波动率
fig_vol = analyzer.plot_rolling_volatility(df_clean)
fig_vol.show()

# ACF/PACF 自相关分析
fig_acf = analyzer.plot_acf_pacf(df_clean)
fig_acf.show()

# ADF 平稳性检验
print('\n📊 ADF 平稳性检验：')
stationarity_results = analyzer.test_stationarity(df_clean)
for series_name, result in stationarity_results.items():
    print(f'\n  {series_name}:')
    for key, val in result.items():
        print(f'    {key}: {val}')

# 正态性检验
print('\n📊 正态性检验：')
normality_results = analyzer.test_normality(df_clean)
for test_name, result in normality_results.items():
    print(f'\n  {test_name}:')
    for key, val in result.items():
        print(f'    {key}: {val}')

## 4. 特征工程

构建机器学习模型所需的多维度特征：

| 特征类别 | 具体指标 |
|---------|----------|
| 技术指标 | MA, EMA, RSI, MACD, 布林带, ATR, ADX, CCI, Williams %R |
| 动量特征 | 多周期收益率, 价格动量, 成交量动量 |
| 波动率特征 | 历史波动率, Parkinson, Garman-Klass |
| 时间特征 | 小时/星期/月份周期编码 |
| 滞后特征 | Lag 1-5, 滚动窗口统计 |

In [ ]:
# ============================================
# 4.1 构建特征集
# ============================================
from feature_engineer import FeatureEngineer

# 初始化特征工程器
fe = FeatureEngineer(scaler_type='standard', models_dir=MODELS_DIR)

# 构建完整特征集
df_features = fe.build_features(df_clean)

# 查看特征列表
print(f'\n📋 特征列表（共 {len(fe.feature_names)} 个）：')
for i, name in enumerate(fe.feature_names, 1):
    print(f'  {i:3d}. {name}')

# 特征相关性分析
corr_matrix, redundant_pairs = fe.get_correlation_matrix(df_features, threshold=0.85)
fig_corr = fe.plot_correlation_heatmap(corr_matrix)
fig_corr.show()

# 准备 ML 数据（特征标准化）
X, y = fe.prepare_ml_data(df_features, label_col='label_direction', scale=True)
print(f'\n📊 特征矩阵形状: {X.shape}')
print(f'   标签分布: {dict(y.value_counts())}')

# 特征重要性预览
fig_imp = fe.plot_feature_importance_preview(X, y)
fig_imp.show()

# 时间序列数据划分
X_train, X_val, X_test, y_train, y_val, y_test = fe.split_time_series(
    X, y, train_ratio=CONFIG['train_ratio'], val_ratio=CONFIG['val_ratio']
)

## 5. 模型训练与评估 - XGBoost

使用 XGBoost 梯度提升树进行价格方向分类预测：
- 使用 Optuna 贝叶斯优化超参数
- Walk-Forward Validation 时间序列交叉验证
- SHAP 值分析特征重要性

In [ ]:
# ============================================
# 5.1 XGBoost 超参数优化与训练
# ============================================
from models.xgb_model import XGBModel

# 初始化 XGBoost 模型
xgb_model = XGBModel(models_dir=MODELS_DIR)

# Optuna 超参数优化（可选，耗时较长）
# 取消注释以执行超参数优化：
# best_params = xgb_model.optimize_hyperparams(
#     X_train, y_train, X_val, y_val,
#     n_trials=CONFIG['xgb_n_trials'], timeout=600
# )

# 训练模型
xgb_history = xgb_model.train(X_train, y_train, X_val, y_val)

# 评估模型
xgb_model.print_evaluation(X_test, y_test, '测试集')

# 过拟合检查
xgb_model.check_overfitting(X_train, y_train, X_test, y_test)

# Walk-Forward 交叉验证
wf_results = xgb_model.walk_forward_validation(X, y, n_splits=5)

# 特征重要性
fig_xgb_imp = xgb_model.plot_feature_importance(top_n=20)
fig_xgb_imp.show()

# SHAP 分析
xgb_model.compute_shap_values(X_test)
xgb_model.plot_shap_summary(X_test, max_display=20)

# ROC 曲线
fig_roc = xgb_model.plot_roc_curve(X_test, y_test)
fig_roc.show()

# 保存模型
xgb_model.save()

## 6. 模型训练与评估 - LSTM

使用 LSTM 长短期记忆网络捕捉时序模式：
- 滑动窗口序列构造（窗口大小 = 24 个 4h 周期 = 4天）
- Dropout 正则化防止过拟合
- EarlyStopping + ReduceLROnPlateau 训练策略

In [ ]:
# ============================================
# 6.1 LSTM 模型训练
# ============================================
from models.lstm_model import LSTMModel

# 初始化 LSTM 模型
lstm_model = LSTMModel(
    window_size=CONFIG['lstm_window_size'],
    models_dir=MODELS_DIR,
    lstm_units=[128, 64],
    dropout_rate=0.3,
    learning_rate=0.001,
    batch_size=64,
    epochs=100
)

# 训练 LSTM
lstm_history = lstm_model.train(X_train, y_train, X_val, y_val)

# 绘制训练历史
fig_lstm_hist = lstm_model.plot_training_history()
fig_lstm_hist.show()

# 评估 LSTM
lstm_metrics = lstm_model.evaluate(X_test, y_test)
print(f'\n📊 LSTM 测试集评估：')
for key, val in lstm_metrics.items():
    print(f'  {key}: {val:.4f}')

# 过拟合检查
lstm_model.check_overfitting(X_train, y_train, X_test, y_test)

# 保存模型
lstm_model.save()

## 7. 模型集成与策略设计

### 集成方法
XGBoost 和 LSTM 预测结果通过加权投票融合：
- XGBoost 权重: 0.5
- LSTM 权重: 0.5

### 交易规则
- **做多**: 融合概率 > 0.6
- **做空**: 融合概率 < 0.4
- **止损**: 2.0 × ATR
- **止盈**: 3.0 × ATR
- **仓位**: Kelly 公式 + ATR 动态调整

In [ ]:
# ============================================
# 7.1 模型集成与信号生成
# ============================================
from strategy import MLStrategy

# 获取两个模型在测试集上的预测概率
xgb_proba = xgb_model.predict_proba(X_test)
lstm_proba, lstm_y_aligned = lstm_model.get_sequence_predictions_aligned(X_test, y_test)

# 对齐索引（LSTM 因滑动窗口会丢失前 window_size 个样本）
common_index = lstm_proba.index
xgb_proba_aligned = pd.Series(xgb_proba, index=X_test.index).loc[common_index]

# 初始化策略
strategy = MLStrategy(CONFIG)

# 计算集成预测概率
ensemble_proba = strategy.ensemble_predict(
    xgb_proba_aligned.values, lstm_proba.values
)

# 将集成概率添加到测试数据中（用于回测）
df_backtest = df_features.loc[common_index].copy()
df_backtest['ensemble_proba'] = ensemble_proba
df_backtest['xgb_proba'] = xgb_proba_aligned.values
df_backtest['lstm_proba'] = lstm_proba.values

print(f'📊 集成预测统计：')
print(f'  数据量: {len(df_backtest)}')
print(f'  做多信号 (>{CONFIG["long_threshold"]}): {(ensemble_proba > CONFIG["long_threshold"]).sum()}')
print(f'  做空信号 (<{CONFIG["short_threshold"]}): {(ensemble_proba < CONFIG["short_threshold"]).sum()}')
print(f'  无信号: {((ensemble_proba >= CONFIG["short_threshold"]) & (ensemble_proba <= CONFIG["long_threshold"])).sum()}')

# 打印策略说明
print(strategy.get_strategy_description())

## 8. 回测执行

使用自研事件驱动型回测引擎，在 2020年至今的历史数据上模拟策略执行：
- 初始资金: 10,000 USDT
- 手续费: 0.1%
- 滑点: 0.05%
- 逐 4h K线模拟执行

In [ ]:
# ============================================
# 8.1 执行回测
# ============================================
from backtester import Backtester

# 初始化回测引擎
backtester = Backtester(strategy=strategy, config=CONFIG)

# 执行回测
results = backtester.run(df_backtest)

# 生成 Buy & Hold 基准曲线
initial_price = df_backtest['close'].iloc[0]
benchmark_curve = (df_backtest['close'] / initial_price) * CONFIG['initial_capital']
benchmark_curve.name = 'Buy & Hold'

# 绘制资金曲线
fig_equity = backtester.plot_equity_curve(benchmark=benchmark_curve)
fig_equity.show()

# 查看交易记录
trades_df = backtester.get_trades_df()
print(f'\n📋 交易记录（前10笔）：')
trades_df.head(10)

# 保存交易记录
backtester.save_trades(os.path.join(RESULTS_DIR, 'trades.csv'))

## 9. 回测结果分析与投资建议

全面评估策略表现：
- 核心绩效指标（夏普比率、最大回撤、Calmar比率等）
- 月度/年度收益分析
- 与 Buy & Hold 基准对比
- 参数敏感性测试
- 模型衰减分析
- 结构化投资建议报告

In [ ]:
# ============================================
# 9.1 绩效分析与投资建议
# ============================================
from analyzer import PerformanceAnalyzer

# 初始化分析器
perf_analyzer = PerformanceAnalyzer(
    equity_curve=results['equity_curve'],
    trades_df=results['trades'],
    benchmark_curve=benchmark_curve,
    initial_capital=CONFIG['initial_capital']
)

# 打印核心绩效指标
perf_analyzer.print_metrics_table()

# 月度收益热力图
fig_monthly = perf_analyzer.plot_monthly_returns_heatmap()
fig_monthly.show()

# 年度收益柱状图
fig_annual = perf_analyzer.plot_annual_returns()
fig_annual.show()

# 回撤曲线（水下曲线）
fig_dd = perf_analyzer.plot_drawdown_curve()
fig_dd.show()

# 策略 vs Buy & Hold 对比
fig_compare = perf_analyzer.compare_with_benchmark()
fig_compare.show()

# 生成投资建议报告
report = perf_analyzer.generate_report()
print(report)

# 保存报告
perf_analyzer.save_report(os.path.join(RESULTS_DIR, 'report.md'))

## 10. 实盘部署指南

### ⚠️ 重要风险提示

1. **本策略仅供学习研究**，不构成投资建议
2. 加密货币市场波动极大，可能导致本金全部损失
3. 历史回测表现不代表未来收益
4. 建议先使用 Paper Trading 模式充分验证

### 部署步骤

1. 在 Binance 创建 API 密钥（仅开启交易权限，关闭提现权限）
2. 将密钥填入 `.env` 文件
3. 修改 `TRADING_MODE` 为 `paper`（模拟盘）或 `live`（实盘）
4. 运行实盘交易脚本

In [ ]:
# # ============================================
# # 10.1 实盘接口演示（Paper Trading 模式）
# # ============================================
# from exchange import create_exchange, PaperTrading

# # 根据配置创建交易接口
# # 默认为 backtest 模式，不会创建实际连接
# # 修改 CONFIG['trading_mode'] = 'paper' 可启用模拟交易

# if CONFIG['trading_mode'] == 'paper':
#     exchange = create_exchange(CONFIG)
    
#     # 查询当前 BTC 价格
#     price = exchange.get_current_price('BTC/USDT')
#     print(f'当前 BTC/USDT 价格: ${price:,.2f}')
    
#     # 查询余额
#     balance = exchange.get_balance()
#     print(f'账户余额: {balance}')
# elif CONFIG['trading_mode'] == 'live':
#     print('⚠️ 实盘模式 - 请确认已充分了解风险！')
#     print('请在 .env 文件中配置 BINANCE_API_KEY 和 BINANCE_SECRET')
#     # exchange = create_exchange(CONFIG)
# else:
#     print('ℹ️ 当前为回测模式，实盘接口未激活。')
#     print('如需启用模拟交易，请修改 .env 中 TRADING_MODE=paper')
#     print('\n📌 实盘部署步骤：')
#     print('  1. 在 Binance 创建 API 密钥（仅开启交易权限）')
#     print('  2. cp .env.example .env && 编辑 .env 填入密钥')
#     print('  3. 修改 TRADING_MODE=paper 先进行模拟交易验证')
#     print('  4. 验证通过后修改 TRADING_MODE=live 启用实盘')